# Notebook 06 — End-to-End Feedback Loop

This notebook demonstrates the complete feedback loop that converts analysis
results into new training data, deduplicates against previously seen entities,
uploads to S3, and triggers a new Flywheel training iteration.

**What this covers:**
- Auto-labelling rules: CONFIRMED → positive, high-confidence NO_MATCH → negative
- REVIEW items → human review queue (not auto-labelled)
- Deduplication via `seen_hashes.txt` to prevent reprocessing the same entities
- `run_feedback()` end-to-end flow

**Why a feedback loop:** A model that only uses its initial training data stagnates.
By continuously feeding confirmed matches and high-confidence negatives back into
Comprehend, the model improves on the specific entity patterns it encounters in
production — a closed-loop system that gets better over time.

In [ ]:
import sys
import os
import tempfile
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from unittest.mock import patch, MagicMock

from ner_ofac.training.feedback_loop import (
    _entity_hash,
    _load_seen_hashes,
    _save_seen_hashes,
    run_feedback,
)

## 1. Auto-Labelling Rules

The feedback loop applies three rules to each analysis result:

| Rule | Condition | Action |
|------|-----------|--------|
| Positive | `routing == CONFIRMED` OR `composite >= 0.90` | → training annotation (entity type label) |
| Negative | `comprehend_score >= 0.85` AND `composite < 0.65` | → negative example |
| Human Review | `routing == REVIEW` (0.65–0.89) | → human review queue file |

Context-expanded CONFIRMED results are also auto-labelled as positive.

In [ ]:
# Build mock analysis results covering all three routing tiers.
mock_analysis_results = [
    {
        'row_index': 0, 'text': 'JOHN DOE was sanctioned.',
        'entity_text': 'JOHN DOE', 'entity_type': 'SANCTIONED_PERSON',
        'matched_name': 'JOHN DOE', 'composite_score': 1.0,
        'comprehend_score': 0.98, 'routing': 'CONFIRMED',
        'BeginOffset': 0, 'EndOffset': 8,
    },
    {
        'row_index': 1, 'text': 'IVAN PETROV was identified.',
        'entity_text': 'IVAN PETROV', 'entity_type': 'SANCTIONED_PERSON',
        'matched_name': 'IVAN PETROV', 'composite_score': 0.75,
        'comprehend_score': 0.80, 'routing': 'REVIEW',
        'BeginOffset': 0, 'EndOffset': 11,
    },
    {
        'row_index': 2, 'text': 'John Smith attended the meeting.',
        'entity_text': 'John Smith', 'entity_type': 'SANCTIONED_PERSON',
        'matched_name': None, 'composite_score': 0.35,
        'comprehend_score': 0.88, 'routing': 'NO_MATCH',  # high confidence negative
        'BeginOffset': 0, 'EndOffset': 10,
    },
    {
        'row_index': 3, 'text': 'ACME CORP was flagged.',
        'entity_text': 'ACME CORP', 'entity_type': 'SANCTIONED_ORG',
        'matched_name': 'ACME CORP', 'composite_score': 0.95,
        'comprehend_score': 0.93, 'routing': 'CONFIRMED',
        'BeginOffset': 0, 'EndOffset': 9,
    },
]

for r in mock_analysis_results:
    print(f"  [{r['routing']:10}] {r['entity_text']:<20} composite={r['composite_score']:.2f}")

## 2. Entity Hashing for Deduplication

Each entity is hashed as `SHA-256(entity_type::entity_text_lowercased)`.
Hashes are stored in `seen_hashes.txt` to prevent the same entity from being
re-submitted to training across multiple pipeline runs.

In [ ]:
# Show hashes for our mock entities.
print(f"{'Entity':<25} {'Type':<25} Hash")
print("-" * 100)
for r in mock_analysis_results:
    h = _entity_hash(r['entity_text'], r['entity_type'])
    print(f"{r['entity_text']:<25} {r['entity_type']:<25} {h[:32]}...")

In [ ]:
# Demonstrate deduplication.
with tempfile.TemporaryDirectory() as tmp_dir:
    hash_path = os.path.join(tmp_dir, 'seen_hashes.txt')
    
    # Pre-populate with JOHN DOE's hash.
    pre_existing_hash = _entity_hash('JOHN DOE', 'SANCTIONED_PERSON')
    _save_seen_hashes([pre_existing_hash], hash_path)
    
    loaded = _load_seen_hashes(hash_path)
    print(f"Pre-existing hashes: {len(loaded)}")
    print(f"JOHN DOE hash present: {pre_existing_hash in loaded}")

## 3. Running the Full Feedback Loop

`run_feedback()` implements the complete pipeline.
We mock S3 and Flywheel calls to run it without real AWS credentials.

In [ ]:
# Build sentences DataFrame.
sentences_df = pd.DataFrame([
    {'row_index': r['row_index'], 'text': r['text'], 'filename': f"doc_{r['row_index']}.txt"}
    for r in mock_analysis_results
])

sentences_df

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    seen_path = os.path.join(tmp_dir, 'seen_hashes.txt')
    output_dir = os.path.join(tmp_dir, 'output')
    
    with patch('ner_ofac.training.feedback_loop.upload_annotations') as mock_upload, \
         patch('ner_ofac.training.feedback_loop.update_flywheel_iteration') as mock_fly:
        
        mock_upload.return_value = 's3://my-bucket/training/feedback/annotations.csv'
        
        new_count = run_feedback(
            mock_analysis_results,
            sentences_df,
            flywheel_arn='arn:aws:comprehend:us-east-1:123456:flywheel/ner-ofac',
            s3_bucket='my-bucket',
            output_dir=output_dir,
            seen_hashes_path=seen_path,
        )
        
        print(f"New annotation rows added: {new_count}")
        print(f"upload_annotations called: {mock_upload.called}")
        print(f"update_flywheel_iteration called: {mock_fly.called}")
        
        # Show output files created.
        if os.path.exists(output_dir):
            print(f"\nOutput files in {output_dir}:")
            for f in os.listdir(output_dir):
                print(f"  - {f}")
        
        # Show seen hashes.
        loaded_hashes = _load_seen_hashes(seen_path)
        print(f"\nSeen hashes after run: {len(loaded_hashes)}")

## 4. Second Run: Deduplication in Action

Running feedback again with the same results should produce 0 new annotations
because all entities are now in `seen_hashes.txt`.

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    seen_path = os.path.join(tmp_dir, 'seen_hashes.txt')
    output_dir = os.path.join(tmp_dir, 'output')
    
    with patch('ner_ofac.training.feedback_loop.upload_annotations') as mock_upload, \
         patch('ner_ofac.training.feedback_loop.update_flywheel_iteration') as mock_fly:
        
        mock_upload.return_value = 's3://my-bucket/training/feedback/annotations.csv'
        
        # First run.
        count1 = run_feedback(mock_analysis_results, sentences_df,
                               flywheel_arn='arn:test', s3_bucket='bucket',
                               output_dir=output_dir, seen_hashes_path=seen_path)
        print(f"First run: {count1} new annotations")
        
        # Reset mock call counts.
        mock_upload.reset_mock()
        mock_fly.reset_mock()
        
        # Second run with same data.
        count2 = run_feedback(mock_analysis_results, sentences_df,
                               flywheel_arn='arn:test', s3_bucket='bucket',
                               output_dir=output_dir, seen_hashes_path=seen_path)
        print(f"Second run: {count2} new annotations (should be 0 — all deduplicated)")
        print(f"S3 upload called on second run: {mock_upload.called} (should be False)")

## Summary

- **CONFIRMED** entities (composite ≥ 0.90) → auto-labelled positive training examples
- **High-confidence negatives** (Comprehend ≥ 0.85, composite < 0.65) → negative examples
- **REVIEW** tier (0.65–0.89) → human review queue file for manual inspection
- **Deduplication** via `seen_hashes.txt` prevents redundant retraining
- On new data: annotations uploaded to S3 and Flywheel iteration triggered
- The loop creates a self-improving system that continuously learns from production data